## Import Dataset

In [4]:
import pandas as pd

# Membaca/memuat dataset hasil penggabungan (cuaca Bandung + harga cabai Jakarta)
df = pd.read_csv('../dataset/dataset_merged1.csv')
df

,tanggal,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
0,2024-05-01,21.8,30.0,26.1,81.0,2.5,5.2,2.0,240.0,83350.0
1,2024-05-02,21.8,32.2,26.1,73.0,5.0,6.0,2.0,120.0,84150.0
2,2024-05-03,22.0,32.0,26.2,73.0,3.9,5.3,2.0,80.0,82500.0
3,2024-05-06,20.8,31.0,25.7,68.0,0.0,7.0,3.0,230.0,91250.0
4,2024-05-07,20.7,32.0,26.2,66.0,3.2,7.4,3.0,300.0,87500.0
...,...,...,...,...,...,...,...,...,...,...
513,2026-04-20,21.2,30.6,24.8,80.0,17.8,7.2,3.0,120.0,61450.0
514,2026-04-21,21.2,30.0,25.1,76.0,6.2,5.8,3.0,286.0,61450.0
515,2026-04-22,21.4,30.2,25.6,79.0,1.5,4.2,3.0,230.0,63100.0
516,2026-04-23,21.8,29.2,24.9,85.0,0.2,4.5,2.0,280.0,64600.0


## Data Cleaning

Pada data gabungan(dataset_merged.csv), data cuaca sudah bersih, tetapi data harga cabai biasanya memiliki kendala khas time series, yaitu adanya tanggal yang lompat (hari libur/akhir pekan), sehingga perlu dilakukan resampling agar data menjadi kontinu setiap hari. langkah-langkah yang dilakukan dalam proses data cleaning ini meliputi:
- Membuat Garis Waktu yang Kontinu: Mengisi tanggal-tanggal kosong (seperti Sabtu & Minggu) agar deret waktunya urut tanpa ada hari yang terlewat.
- Imputasi Nilai Kosong (Imputation): Mengisi harga cabai pada tanggal libur tersebut menggunakan metode time series seperti forward fill (mengikuti harga hari terakhir sebelum libur) atau linear interpolation.

In [5]:
# 1. Pastikan kolom tanggal bertipe datetime
df['tanggal'] = pd.to_datetime(df['tanggal'])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   tanggal            518 non-null    datetime64[ns]
 1   TN                 518 non-null    float64       
 2   TX                 518 non-null    float64       
 3   TAVG               518 non-null    float64       
 4   RH_AVG             518 non-null    float64       
 5   RR                 518 non-null    float64       
 6   SS                 518 non-null    float64       
 7   FF_X               518 non-null    float64       
 8   DDD_X              518 non-null    float64       
 9   Cabai Merah Besar  518 non-null    float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 40.6 KB


In [9]:
#Menampilkan 5 baris pertama untuk memastikan data sudah benar
df.head()

,tanggal,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
0,2024-05-01,21.8,30.0,26.1,81.0,2.5,5.2,2.0,240.0,83350.0
1,2024-05-02,21.8,32.2,26.1,73.0,5.0,6.0,2.0,120.0,84150.0
2,2024-05-03,22.0,32.0,26.2,73.0,3.9,5.3,2.0,80.0,82500.0
3,2024-05-06,20.8,31.0,25.7,68.0,0.0,7.0,3.0,230.0,91250.0
4,2024-05-07,20.7,32.0,26.2,66.0,3.2,7.4,3.0,300.0,87500.0


In [10]:
#Mengurutkan berdasarkan tanggal
df = df.sort_values('tanggal').reset_index(drop=True)
df

,tanggal,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
0,2024-05-01,21.8,30.0,26.1,81.0,2.5,5.2,2.0,240.0,83350.0
1,2024-05-02,21.8,32.2,26.1,73.0,5.0,6.0,2.0,120.0,84150.0
2,2024-05-03,22.0,32.0,26.2,73.0,3.9,5.3,2.0,80.0,82500.0
3,2024-05-06,20.8,31.0,25.7,68.0,0.0,7.0,3.0,230.0,91250.0
4,2024-05-07,20.7,32.0,26.2,66.0,3.2,7.4,3.0,300.0,87500.0
...,...,...,...,...,...,...,...,...,...,...
513,2026-04-20,21.2,30.6,24.8,80.0,17.8,7.2,3.0,120.0,61450.0
514,2026-04-21,21.2,30.0,25.1,76.0,6.2,5.8,3.0,286.0,61450.0
515,2026-04-22,21.4,30.2,25.6,79.0,1.5,4.2,3.0,230.0,63100.0
516,2026-04-23,21.8,29.2,24.9,85.0,0.2,4.5,2.0,280.0,64600.0


In [11]:
#Set tanggal sebagai index untuk proses resampling
df.set_index('tanggal', inplace=True)


In [12]:
df

,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
tanggal,,,,,,,,,
2024-05-01,21.8,30.0,26.1,81.0,2.5,5.2,2.0,240.0,83350.0
2024-05-02,21.8,32.2,26.1,73.0,5.0,6.0,2.0,120.0,84150.0
2024-05-03,22.0,32.0,26.2,73.0,3.9,5.3,2.0,80.0,82500.0
2024-05-06,20.8,31.0,25.7,68.0,0.0,7.0,3.0,230.0,91250.0
2024-05-07,20.7,32.0,26.2,66.0,3.2,7.4,3.0,300.0,87500.0
...,...,...,...,...,...,...,...,...,...
2026-04-20,21.2,30.6,24.8,80.0,17.8,7.2,3.0,120.0,61450.0
2026-04-21,21.2,30.0,25.1,76.0,6.2,5.8,3.0,286.0,61450.0
2026-04-22,21.4,30.2,25.6,79.0,1.5,4.2,3.0,230.0,63100.0


In [ ]:
#Resample menjadi harian ('D') agar tidak ada tanggal yang lompat/bolong
#Gunakan limit_direction='forward' untuk mengisi cuaca dan harga yang kosong karena resampling
df_clean = df.resample('D').interpolate(method='linear')

In [14]:
#Kembalikan kolom tanggal dari index
df_clean = df_clean.reset_index()
print("Jumlah baris setelah Data Cleaning (kontinu):", len(df_clean))
df_clean

Jumlah baris setelah Data Cleaning (kontinu): 724


,tanggal,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
0,2024-05-01,21.8,30.000000,26.100000,81.000000,2.5,5.200000,2.000000,240.0,83350.000000
1,2024-05-02,21.8,32.200000,26.100000,73.000000,5.0,6.000000,2.000000,120.0,84150.000000
2,2024-05-03,22.0,32.000000,26.200000,73.000000,3.9,5.300000,2.000000,80.0,82500.000000
3,2024-05-04,21.6,31.666667,26.033333,71.333333,2.6,5.866667,2.333333,130.0,85416.666667
4,2024-05-05,21.2,31.333333,25.866667,69.666667,1.3,6.433333,2.666667,180.0,88333.333333
...,...,...,...,...,...,...,...,...,...,...
719,2026-04-20,21.2,30.600000,24.800000,80.000000,17.8,7.200000,3.000000,120.0,61450.000000
720,2026-04-21,21.2,30.000000,25.100000,76.000000,6.2,5.800000,3.000000,286.0,61450.000000
721,2026-04-22,21.4,30.200000,25.600000,79.000000,1.5,4.200000,3.000000,230.0,63100.000000
722,2026-04-23,21.8,29.200000,24.900000,85.000000,0.2,4.500000,2.000000,280.0,64600.000000


### Penjelasan Data Cleaning
Data runtun waktu asli memiliki jeda waktu (*gaps*) karena tidak adanya pencatatan harga pangan pada hari Sabtu, Minggu, atau libur nasional. Model berbasis deret waktu memerlukan kontinuitas tanggal yang teratur. Langkah-langkah yang dilakukan meliputi:
* **Penyelarasan Indeks Tanggal:** Mengubah kolom `tanggal` menjadi tipe data `datetime` dan menetapkannya sebagai indeks utama dataframe.
* **Resampling Harian (`resample('D')`):** Membuat ulang baris tanggal agar berurutan secara harian tanpa ada hari yang melompat.
* **Interpolasi Linear (`interpolate(method='linear')`):** Mengisi kekosongan nilai harga (*missing values*) pada hari libur menggunakan pendekatan interpolasi linear, serta menggunakan metode pengisian maju (*forward fill*) untuk variabel cuaca konstan demi menjaga kontinuitas data tanpa merusak tren asli.

## Feature Extraction

Setelah data benar-benar bersih dan kontinu setiap hari, barulah kita mengekstrak fitur baru berdasarkan insight emas yang Anda temukan di tahap EDA tadi (bahwa cuaca Bandung 30-45 hari lalu sangat memengaruhi harga hari ini).

Fitur-fitur yang wajib Anda ekstrak untuk Model Prediksi Cabai:
- Fitur Jeda Waktu (Lag Features): Menggeser data cuaca Bandung ke masa lalu.
- Fitur Rata-rata Bergerak (Rolling Features): Mengakumulasikan kondisi cuaca seminggu/dua minggu terakhir untuk melihat tren iklim.
- Fitur Kalender (Time Features): Ekstrak bulan dan hari untuk membantu model mengenali pola musiman (misal: setiap menjelang Lebaran atau Tahun Baru harga cabai selalu naik).

In [15]:
df_features = df_clean.copy()

In [16]:
# A. Membuat Fitur Lag (Berdasarkan hasil EDA Anda di Lag 30 dan 45)

# Lag Curah Hujan (RR) dan Kelembapan (RH_AVG) 30 & 45 hari lalu
df_features['RR_lag_30'] = df_features['RR'].shift(30)
df_features['RR_lag_45'] = df_features['RR'].shift(45)

df_features['RH_lag_30'] = df_features['RH_AVG'].shift(30)
df_features['RH_lag_45'] = df_features['RH_AVG'].shift(45)

In [17]:
# B. Membuat Fitur Rolling (Rata-rata akumulasi cuaca 14 hari terakhir)
df_features['RR_rolling_mean_14'] = df_features['RR'].rolling(window=14).mean()
df_features['RH_rolling_mean_14'] = df_features['RH_AVG'].rolling(window=14).mean()

In [18]:
# C. Fitur Historis Harga Target (Pemicu akurasi utama runtun waktu)
df_features['Cabai_lag_1'] = df_features['Cabai Merah Besar'].shift(1)  # Harga cabai kemarin
df_features['Cabai_lag_7'] = df_features['Cabai Merah Besar'].shift(7)  # Harga cabai minggu lalu

In [19]:
# D. Fitur Kalender Musiman
df_features['bulan'] = df_features['tanggal'].dt.month

In [20]:
# E. Seleksi Kolom Penting: Hapus kolom cuaca mentah hari ini untuk menghindari tumpang tindih (Multikolinearitas)
kolom_pilihan = [
    'tanggal', 'Cabai Merah Besar',
    'Cabai_lag_1', 'Cabai_lag_7',
    'RR_lag_45', 'RH_lag_30',
    'RR_rolling_mean_14', 'bulan'
]
df_features = df_features[kolom_pilihan]

In [21]:
# F. Hapus baris NaN akibat pergeseran waktu (lag)
df_features = df_features.dropna().reset_index(drop=True)

In [22]:
# Tampilkan hasil dataframe siap pakai untuk modeling
df_features

,tanggal,Cabai Merah Besar,Cabai_lag_1,Cabai_lag_7,RR_lag_45,RH_lag_30,RR_rolling_mean_14,bulan
0,2024-06-15,71650.0,71650.000000,65550.0,2.5,70.000000,6.848810,6
1,2024-06-16,71650.0,71650.000000,64450.0,5.0,76.000000,6.346429,6
2,2024-06-17,71650.0,71650.000000,63350.0,3.9,74.000000,5.592857,6
3,2024-06-18,71650.0,71650.000000,65000.0,2.6,72.000000,3.092857,6
4,2024-06-19,77500.0,71650.000000,65000.0,1.3,70.000000,1.426190,6
...,...,...,...,...,...,...,...,...
674,2026-04-20,61450.0,59966.666667,56200.0,1.4,75.333333,12.364286,4
675,2026-04-21,61450.0,61450.000000,56200.0,2.7,71.666667,12.685714,4
676,2026-04-22,63100.0,61450.000000,57000.0,4.0,68.000000,11.900000,4
677,2026-04-23,64600.0,63100.000000,57000.0,5.3,69.000000,11.650000,4


## Penjelasan Feature Extraction
Merujuk pada temuan di tahap EDA mengenai adanya efek tunda (*time-lag effect*) cuaca daerah pemasok terhadap harga di Jakarta, dibuatlah beberapa fitur prediktor baru:

* **Fitur Jeda Waktu (*Lag Features*):** Membuat kolom cuaca masa lalu dengan menggeser data (`.shift(30)` dan `.shift(45)`). Fitur utama yang dibuat adalah `RR_lag_30`, `RR_lag_45` (Curah Hujan) serta `RH_lag_30`, `RH_lag_45` (Kelembapan Udara). Fitur ini membantu model memahami kondisi iklim Bandung pada 1 hingga 1,5 bulan yang lalu.
* **Fitur Rata-Rata Bergerak (*Rolling Features*):** Menghitung akumulasi rata-rata cuaca dalam jendela waktu 14 hari terakhir (`.rolling(window=14).mean()`) untuk fitur Curah Hujan, Kelembapan. Fitur ini berfungsi untuk memuluskan (*smoothing*) fluktuasi cuaca harian yang terlalu acak dan ekstrem, sehingga model bisa menangkap tren iklim jangka pendek.
* **Fitur Kalender (*Time-Based Features*):** Mengekstrak komponen `bulan` dari kolom tanggal untuk memfasilitasi model dalam mempelajari pola kenaikan harga musiman tahunan (seperti menjelang hari raya besar atau musim kemarau/hujan ekstrem).

Selain itu kita melakukan Penanganan Akhir Data (*Final Drop*) pada Bagian Feature Extraction ini

Proses pergeseran waktu (*lagging*) dan rata-rata bergerak (*rolling*) secara otomatis akan menghasilkan nilai kosong (`NaN`) pada baris-baris awal dataset (sebanyak rentang jendela terbesar, yaitu 45 baris pertama). Seluruh baris yang mengandung nilai kosong tersebut dihapus menggunakan fungsi `.dropna()` untuk menghasilkan dataset akhir yang bersih, kontinu, dan siap digunakan pada tahap pemisahan data (*Train-Test Split*) serta pemodelan.

In [23]:
output_path = '../dataset/dataset_merge_cleaning.csv'
df_features.to_csv(output_path, index=False)
print(f"\nDataset berhasil disimpan ke {output_path}")


Dataset berhasil disimpan ke ../dataset/dataset_merge_cleaning.csv
